In [3]:
from pathlib import Path
import json
import os
import sys
import subprocess
import importlib

REQ_FILE = Path("requirements.txt")
if REQ_FILE.exists():
    try:
        import torch
    except Exception:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", str(REQ_FILE)])
if os.name == "nt":
    try:
        import torch_directml
    except Exception:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "torch-directml"])

PIPELINE_VERSION = "schemeA-static-mapping-v1"
print(f"[main] pipeline_version={PIPELINE_VERSION}", flush=True)
PROJECT_ROOT = Path.cwd()
print(f"[main] cwd={PROJECT_ROOT}", flush=True)
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import food_cv.training as training_module
importlib.reload(training_module)
from food_cv.classifier import build_resnet50_classifier, unfreeze_last_two_blocks
from food_cv.config import ProjectPaths
from food_cv.data_pipeline import DataConfig, Food101DataModule
from food_cv.evaluation import evaluate_classification_metrics, evaluate_nutrition_hit_rate
from food_cv.pipeline import MealPredictor
from food_cv.training import TrainConfig, train_classifier_two_stage

paths = ProjectPaths.from_root(PROJECT_ROOT)
demo_image = PROJECT_ROOT / "demo.jpg"

candidate_data_roots = [
    PROJECT_ROOT / "data",
    PROJECT_ROOT / "datasets",
    PROJECT_ROOT / "food101_data",
    Path("/content/data"),
    Path("/content/datasets"),
    PROJECT_ROOT,
]
data_root = next((p for p in candidate_data_roots if p.exists()), PROJECT_ROOT)
data_config = DataConfig(data_root=data_root, batch_size=16, num_workers=0, image_size=224, download_if_missing=True)

print("[main] 初始化数据模块", flush=True)
try:
    datamodule = Food101DataModule(data_config)
    print("[main] 构建 DataLoader", flush=True)
    train_loader, val_loader, test_loader = datamodule.build_dataloaders()
    class_names = datamodule.get_class_names()
    print(f"train={len(train_loader.dataset)} val={len(val_loader.dataset)} test={len(test_loader.dataset)}")
    print(f"labels={len(class_names)}")
except Exception as e:
    train_loader = val_loader = test_loader = None
    class_names = []
    print(f"Data loader 初始化失败: {e}")

model = build_resnet50_classifier(num_classes=101, freeze_backbone=True)
stage1_checkpoint = paths.model_dir / "food101_resnet50_stage1.pt"
stage2_checkpoint = paths.model_dir / "food101_resnet50_stage2.pt"

if train_loader is not None and val_loader is not None:
    print("[main] 开始训练（两阶段：stage1->stage2）", flush=True)
    metrics = train_classifier_two_stage(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        stage1_save_path=stage1_checkpoint,
        stage1_config=TrainConfig(epochs=1, lr=1e-4, device="directml" if os.name == "nt" else "auto", log_every_n_steps=5, max_steps_per_epoch=60, use_amp=True, require_accelerator=True, optimizer_foreach=False),
        stage2_save_path=stage2_checkpoint,
        stage2_config=TrainConfig(epochs=1, lr=3e-5, device="directml" if os.name == "nt" else "auto", log_every_n_steps=5, max_steps_per_epoch=60, use_amp=True, require_accelerator=True, optimizer_foreach=False),
        unfreeze_fn=unfreeze_last_two_blocks,
    )
    print(json.dumps(metrics, ensure_ascii=False, indent=2))
    if not isinstance(metrics, dict) or "stage1" not in metrics or "stage2" not in metrics:
        raise RuntimeError("当前训练返回不是两阶段结构，疑似运行到旧代码路径。请重启内核后 Run All。")
    print(f"[main] stage1_checkpoint_exists={stage1_checkpoint.exists()}")
    print(f"[main] stage2_checkpoint_exists={stage2_checkpoint.exists()}")
    quick_eval = evaluate_classification_metrics(model=model, loader=test_loader, device=model.fc.weight.device, max_batches=80)
    print("[main] quick test metrics (sampled):")
    print(json.dumps(quick_eval, ensure_ascii=False, indent=2))
else:
    print("跳过训练：请确认 Food-101 已下载并放到 data 目录")

inference_checkpoint = stage2_checkpoint if stage2_checkpoint.exists() else (stage1_checkpoint if stage1_checkpoint.exists() else None)
if inference_checkpoint is not None and demo_image.exists():
    if inference_checkpoint == stage1_checkpoint:
        print("[main] 警告：Stage2 checkpoint 缺失，当前回退使用 Stage1 checkpoint 进行推理")
    predictor = MealPredictor(paths=paths, checkpoint_path=inference_checkpoint, labels=class_names if class_names else None)
    result = predictor.predict_meal(demo_image)
    print(json.dumps(result, ensure_ascii=False, indent=2))
    nutrition_eval = evaluate_nutrition_hit_rate(predictor, [demo_image])
    print("[main] nutrition hit metrics:")
    print(json.dumps(nutrition_eval, ensure_ascii=False, indent=2))
    if result.get("total", {}).get("calories", 0.0) <= 0.0:
        print("[main] 警告：营养命中率偏低，请继续扩展 Food-101->USDA 映射", flush=True)
elif inference_checkpoint is None:
    print("未找到可用 checkpoint（stage1/stage2 均缺失），已跳过推理。")
else:
    print("未找到 demo.jpg，推理演示已跳过")


[main] pipeline_version=solo-stage2-hardfix-v4
[main] cwd=d:\CityU\Projects\CV_course_project
[main] eval_json_path=D:\CityU\Projects\CV_course_project\eval_samples.json
[main] 初始化数据模块
[main] 构建 DataLoader
train=75750 val=25250 test=25250
labels=101
[main] 开始训练（两阶段：stage1->stage2）
[train] two-stage training: stage1(frozen) start
[train] backend=directml device=privateuseone:0
[train] epoch=1/1 start
[train] epoch_loop start total_steps=60
[train] first_batch_loaded
[train] step=5/60 avg_loss=4.6292 speed=9.10step/s eta=0.1min
[train] step=10/60 avg_loss=4.6361 speed=8.93step/s eta=0.1min
[train] step=15/60 avg_loss=4.6332 speed=8.95step/s eta=0.1min
[train] step=20/60 avg_loss=4.6323 speed=9.02step/s eta=0.1min
[train] step=25/60 avg_loss=4.6207 speed=9.01step/s eta=0.1min
[train] step=30/60 avg_loss=4.6249 speed=8.97step/s eta=0.1min
[train] step=35/60 avg_loss=4.6236 speed=8.99step/s eta=0.0min
[train] step=40/60 avg_loss=4.6232 speed=8.99step/s eta=0.0min
[train] step=45/60 avg_loss

In [13]:
from google.colab import drive
import os
from pathlib import Path

drive.mount('/content/drive')

GITHUB_TOKEN = os.getenv('GITHUB_TOKEN', '')
REPO_OWNER = 'Blank4cc'
REPO_NAME = 'VisualDietician'
if not GITHUB_TOKEN:
    raise RuntimeError('请先在 Colab 环境变量中设置 GITHUB_TOKEN')
PRIVATE_REPO_URL = f'https://{GITHUB_TOKEN}@github.com/{REPO_OWNER}/{REPO_NAME}.git'
DEST_PATH = f'/content/drive/MyDrive/{REPO_NAME}'

if not os.path.exists(DEST_PATH):
    print('正在尝试克隆私有仓库...')
    !git clone {PRIVATE_REPO_URL} {DEST_PATH}
else:
    print(f'目录已存在: {DEST_PATH}')

if os.path.exists(DEST_PATH):
    os.chdir(DEST_PATH)
    print(f'成功进入目录: {os.getcwd()}')
    !ls
else:
    print('克隆仍然失败，请检查 Token 权限或仓库路径是否正确。')
Path(DEST_PATH)

ModuleNotFoundError: No module named 'google'

In [7]:
import os

# 确保你在项目目录下
if os.path.exists(DEST_PATH):
    os.chdir(DEST_PATH)
    # 使用 git pull 拉取最新代码
    !git pull
    print("代码已尝试更新。")
else:
    print(f"未找到目录 {DEST_PATH}，请先运行克隆单元格。")

Already up to date.
代码已尝试更新。


In [8]:
import sys
from pathlib import Path

# 将当前路径（包含 src 目录的路径）加入 sys.path
PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

print("当前 Python 路径已更新，现在可以尝试运行之前的代码单元格了。")

当前 Python 路径已更新，现在可以尝试运行之前的代码单元格了。
